# 🚀 Advanced PySpark Performance Tuning & Optimization (Part 6)
This notebook completes our masterclass optimization series by addressing advanced architectural, serialization, storage, and cluster-level performance techniques: Higher-Order Array Functions, Kryo Serialization, Coalesce vs Repartition Compaction, Disk Spill Mitigation, and Two-Stage Salted Group-By Aggregations.

---
## 21. Native Higher-Order Array Expressions (`transform`, `filter`, `aggregate`)
### **Concept & Architectural Explanation**
* **The Anti-Pattern:** Exploding arrays into rows via `F.explode()`, applying row operations, and re-grouping back via `F.collect_list()` causes expensive network shuffles and massive data expansion.
* **The Optimization:** Native higher-order functions (`F.transform()`, `F.filter()`, `F.aggregate()`) evaluate complex array mutations directly in-place within the underlying Tungsten memory buffer inside JVM WholeStageCodegen loops without exploding data structure rows.
* **Production Best Practice:** Always prefer native higher-order array functions over `explode` + `groupBy` patterns when processing arrays.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Optimization_21_Higher_Order_Functions") \
    .master("local[*]") \
    .getOrCreate()

# Sample data with array of numerical scores
data = [(1, [10, 20, 30, 40]), (2, [15, 25, 35, 45])]
df_array = spark.createDataFrame(data, ["user_id", "scores"])

# Apply in-place array element multiplication and filtering
df_transformed = df_array.withColumn(
    "boosted_scores", 
    F.transform(F.col("scores"), lambda x: x * 2)
).withColumn(
    "high_scores", 
    F.filter(F.col("scores"), lambda x: x > 20)
)

print("=== Physical Plan: Native Higher-Order Array Processing ===")
df_transformed.explain(True)
df_transformed.show(truncate=False)

---
## 22. Out-of-Core Disk Spill & GC Pause Mitigation
### **Concept & Architectural Explanation**
* **What is Spill?** When execution memory for a task is insufficient during wide aggregations or sorts, Spark spills intermediate in-memory data structures to local disk (`Spill (Memory)` vs `Spill (Disk)` in the Spark UI).
* **The Cost:** Disk spill introduces severe disk I/O latency and CPU serialization overhead.
* **The Optimization:** Increase executor memory per core (`spark.executor.memory`) or adjust `spark.memory.fraction` (e.g., from `0.6` to `0.8`) to expand execution space and eliminate disk spilling.

In [ ]:
# Verify active spill and memory parameters
print("Execution Memory Fraction:", spark.conf.get("spark.memory.fraction", "0.6"))
print("Safety override for memory spill configuration verified.")

---
## 23. Kryo Serialization Tuning (`spark.serializer`)
### **Concept & Architectural Explanation**
* **The Standard Serializer:** By default, RDD shuffles and object caching use Java's standard `ObjectOutputStream`, which produces bulky serialized representations and high CPU overhead.
* **Why Kryo is Better:** `KryoSerializer` reduces binary serialized data size up to 10x and speeds up serialization throughput significantly.
* **Production Best Practice:** Set `spark.serializer = org.apache.spark.serializer.KryoSerializer` and register custom domain classes via `spark.kryo.registrator`.

In [ ]:
# Set Kryo Serializer for cluster operations
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
print("Active Serializer:", spark.conf.get("spark.serializer"))

---
## 24. Coalesce vs. Repartition for Target File Sizing
### **Concept & Architectural Explanation**
* **`repartition(N)`:** Triggers a full network `Exchange` shuffle across all cluster nodes to rebalance data uniformly into N partitions.
* **`coalesce(N)`:** Merges adjacent local partitions on the same worker without a network shuffle. **Only use `coalesce` when reducing partition counts** (e.g., compaction before saving to storage).
* **Production Anti-Pattern:** Never call `.coalesce()` before heavy filters or transformations, as it collapses parallelism early and forces a single thread to evaluate upstream operations.

In [ ]:
# Demonstration of coalesce (no shuffle) vs repartition (full shuffle)
df_large = spark.range(0, 1000000).repartition(100)

# Coalesce collapses local partitions with NO Exchange operator
df_coalesced = df_large.coalesce(10)
print("=== Coalesce Physical Plan (No Shuffle Exchange) ===")
df_coalesced.explain(False)

# Repartition introduces explicit Exchange hashpartitioning
df_repartitioned = df_large.repartition(10)
print("=== Repartition Physical Plan (Requires Network Shuffle) ===")
df_repartitioned.explain(False)

---
## 25. Two-Stage Aggregation Salting for Skewed Group-By Keys
### **Concept & Architectural Explanation**
* **The Bottleneck:** When performing `.groupBy("hot_key").agg(...)` on heavily skewed keys (e.g., millions of records for a single `tenant_id`), a single executor node gets overwhelmed computing the final aggregate.
* **The Fix (Two-Stage Partial Aggregation):** 
  1. Stage 1: Add a random salt (`0` to `N-1`) to the group-by key and perform a partial aggregation on `(key, salt)`.
  2. Stage 2: Strip the salt and perform the final aggregation on `key`.
* **The Benefit:** Parallelizes the heavy partial aggregate work across N executors before combining final counts.

In [ ]:
# Step 1: Create heavily skewed group-by dataset
df_skewed_group = spark.range(0, 1000000).withColumn(
    "skewed_tenant", F.when(F.rand() < 0.8, "TENANT_HOT").otherwise(F.concat_ws("_", F.lit("TENANT"), (F.rand()*10).cast("int")))
)

# Step 2: Two-Stage Salted Aggregation
SALT_GROUPS = 4
df_salted_stage1 = df_skewed_group.withColumn("salt", (F.rand() * SALT_GROUPS).cast("int")) \
                                  .groupBy("skewed_tenant", "salt") \
                                  .agg(F.count("*").alias("partial_count"))

df_final_salted_agg = df_salted_stage1.groupBy("skewed_tenant") \
                                      .agg(F.sum("partial_count").alias("total_count"))

print("=== Physical Plan: Two-Stage Salted Group-By Aggregation ===")
df_final_salted_agg.explain(True)

spark.stop()